# Imports

In [1]:
import os, json, time
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
from torchvision import models, transforms
from transformers import (
    ViTModel, ViTImageProcessor,
    AutoModel, AutoImageProcessor,
    CLIPVisionModel, CLIPProcessor
)

from captum.attr import IntegratedGradients
from sklearn.metrics.pairwise import cosine_similarity

/home/aysel/tfe/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [ ]:
from paths import vision_emb_path

# Config

In [2]:
# Paths
CURRENT_DATASET = "Flickr8k"
BASE_DIR = "TFE_Data"
DATASETS_DIR = os.path.join(BASE_DIR, "Datasets")
IMAGE_DIR = os.path.join(BASE_DIR, "Flickr8k", "Images", "Flicker8k_Dataset")

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Using device:", device)

Using device: cuda


# Stress Test

In [3]:
STRESS_TEST_IMAGES = [
    "107582366_d86f2d3347.jpg", 
    "127450902_533ceeddfc.jpg", 
    "112178718_87270d9b4d.jpg",
    "141755290_4b954529f3.jpg", 
    "171488318_fb26af58e2.jpg", 
    "240583223_e26e17ee96.jpg",
    "241346317_be3f07bd2e.jpg", 
    "911795495_342bb15b97.jpg", 
    "2452686995_621878f561.jpg",
    "97577988_65e2eae14a.jpg"
]

stress_paths = [os.path.join(IMAGE_DIR, f) for f in STRESS_TEST_IMAGES]

df_path = os.path.join(DATASETS_DIR, f"df_{CURRENT_DATASET}.pkl")
df = pd.read_pickle(df_path)
print("Loaded dataframe with", len(df), "images.")

IMAGE_PATHS = df["image_path"].tolist()
IMAGE_PATHS[0]



Loaded dataframe with 8091 images.


'/home/aysel/tfe/TFE_Data/Flickr8k/Images/Flicker8k_Dataset/1000268201_693b08cb0e.jpg'

# Load Embedding

In [4]:
VISION_MODELS = ["resnet50", "mobilenet_v3", "vit", "pvt", "clip_vision"]

def load_embeddings(model_name):
    path = f"{BASE_DIR}/Unimodal_Results/Flickr8k/vision/{model_name}/embeddings.npy"
    X = np.load(path)
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)



# Models

In [5]:
def load_indexation_model(model_name):
    if model_name == "resnet50":
        weights = models.ResNet50_Weights.DEFAULT
        model = models.resnet50(weights=weights)
        model.fc = nn.Identity()
        transform = weights.transforms()
        return model.to(device).eval(), transform

    if model_name == "mobilenet_v3":
        weights = models.MobileNet_V3_Large_Weights.DEFAULT
        model = models.mobilenet_v3_large(weights=weights)
        model.classifier = nn.Identity()
        transform = weights.transforms()
        return model.to(device).eval(), transform

    if model_name == "vit":
        processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")
        backbone = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
        def transform(img):
            return processor(images=img, return_tensors="pt")["pixel_values"].squeeze(0)
        return backbone.to(device).eval(), transform

    if model_name == "pvt":
        processor = AutoImageProcessor.from_pretrained("Zetatech/pvt-tiny-224")
        backbone = AutoModel.from_pretrained("Zetatech/pvt-tiny-224")
        def transform(img):
            return processor(images=img, return_tensors="pt")["pixel_values"].squeeze(0)
        return backbone.to(device).eval(), transform

    if model_name == "clip_vision":
        processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
        backbone = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")
        def transform(img):
            return processor(images=img, return_tensors="pt")["pixel_values"].squeeze(0)
        return backbone.to(device).eval(), transform


## ResNet50

In [6]:
def get_resnet50_model(device):
    weights = models.ResNet50_Weights.DEFAULT
    model = models.resnet50(weights=weights)
    model.fc = nn.Linear(2048, 1000)
    return model.to(device).eval(), weights.transforms()


## MobileNetV3

In [7]:
def get_mobilenet_v3_model(device):
    weights = models.MobileNet_V3_Large_Weights.DEFAULT
    model = models.mobilenet_v3_large(weights=weights)
    model.classifier = nn.Linear(960, 1000)
    return model.to(device).eval(), weights.transforms()


## ViT

In [8]:
class ViTWithHead(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.backbone = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
        self.head = nn.Linear(self.backbone.config.hidden_size, 1000)

    def forward(self, x):
        out = self.backbone(pixel_values=x)
        cls = out.last_hidden_state[:, 0]
        return self.head(cls)

def get_vit_model(device):
    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    return ViTWithHead(device).to(device).eval(), transform


## PVT

In [9]:
class CaptumWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        # HuggingFace models expect pixel_values=...
        outputs = self.model(pixel_values=x)
        return outputs.logits


In [10]:
class PVTWithHead(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.backbone = AutoModel.from_pretrained("Zetatech/pvt-tiny-224").to(device).eval()

        # PVT uses embed_dims list, last stage = CLS dimension
        hidden = self.backbone.config.embed_dims[-1]

        self.head = nn.Linear(hidden, 1000).to(device)

    def forward(self, x):
        out = self.backbone(pixel_values=x)
        cls = out.last_hidden_state[:, 0, :]   # CLS token
        return self.head(cls)

from transformers import AutoModelForImageClassification, AutoImageProcessor

def get_pvt_model(device):
    processor = AutoImageProcessor.from_pretrained("Zetatech/pvt-tiny-224")
    base_model = AutoModelForImageClassification.from_pretrained(
        "Zetatech/pvt-tiny-224"
    ).to(device).eval()

    model = CaptumWrapper(base_model)

    def transform(img):
        return processor(images=img, return_tensors="pt")["pixel_values"].squeeze(0)

    return model, transform




## Clip Vision

In [11]:
class CLIPWithHead(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.backbone = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")
        self.head = nn.Linear(self.backbone.config.hidden_size, 1000)

    def forward(self, x):
        out = self.backbone(pixel_values=x)
        pooled = out.pooler_output
        return self.head(pooled)

def get_clip_vision_model(device):
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    def transform(img):
        return processor(images=img, return_tensors="pt")["pixel_values"].squeeze(0)

    return CLIPWithHead(device).to(device).eval(), transform


# Retrieval

In [12]:
def compute_ig_for_image(xai_model, xai_transform, image_path):
    xai_model = xai_model.to(device)
    ig = IntegratedGradients(xai_model)

    img = xai_transform(Image.open(image_path)).unsqueeze(0).to(device)
    attr = ig.attribute(img, target=0)

    heatmap = attr.squeeze().detach().cpu().numpy().mean(axis=0)
    overlay = overlay_heatmap(Image.open(image_path), heatmap)

    del img, attr, ig
    torch.cuda.empty_cache()

    return heatmap, overlay


In [13]:
def show_retrieved_ig(retrieved_paths, retrieved_heatmaps, retrieved_overlays, sims):
    fig, axes = plt.subplots(3, 5, figsize=(22, 12))
    fig.suptitle("IG for Retrieved Images", fontsize=16)

    # Row 1: images
    for i in range(5):
        axes[0, i].imshow(Image.open(retrieved_paths[i]))
        axes[0, i].set_title(f"Top {i+1}\nSim: {sims[i]:.3f}")
        axes[0, i].axis("off")

    # Row 2: heatmaps
    for i in range(5):
        axes[1, i].imshow(retrieved_heatmaps[i], cmap="hot")
        axes[1, i].axis("off")

    # Row 3: overlays
    for i in range(5):
        axes[2, i].imshow(retrieved_overlays[i])
        axes[2, i].axis("off")

    plt.tight_layout()
    plt.show()


In [14]:
def show_topk_retrievals(retrieved_paths, sims, save_path=None):
    fig, axes = plt.subplots(1, 5, figsize=(22, 5))
    fig.suptitle("Top‑5 Retrieved Images", fontsize=16)

    for i in range(5):
        axes[i].imshow(Image.open(retrieved_paths[i]))
        axes[i].set_title(f"Top {i+1}\nSim: {sims[i]:.3f}")
        axes[i].axis("off")

    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")

    plt.show()


In [15]:
def show_query_heatmap_overlay(model_name, query_path, heatmap, overlay, save_path=None):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(f"{model_name} — Query & Explanations", fontsize=16)

    axes[0].imshow(Image.open(query_path))
    axes[0].set_title("Query Image")
    axes[0].axis("off")

    axes[1].imshow(heatmap, cmap="hot")
    axes[1].set_title("IG Heatmap")
    axes[1].axis("off")

    axes[2].imshow(overlay)
    axes[2].set_title("Overlay")
    axes[2].axis("off")

    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")

    plt.show()


In [16]:
def compare_models_ig_across_models(query_path, models_xai, out_dir):
    fig, axes = plt.subplots(1, len(models_xai), figsize=(5 * len(models_xai), 5))
    fig.suptitle("IG Comparison Across Models", fontsize=18)

    for i, (model_name, (xm, xt)) in enumerate(models_xai.items()):
        xm = xm.to(device)
        ig = IntegratedGradients(xm)

        img = xt(Image.open(query_path)).unsqueeze(0).to(device)
        attr = ig.attribute(img, target=0)
        heatmap = attr.squeeze().detach().cpu().numpy().mean(axis=0)

        axes[i].imshow(heatmap, cmap="hot")
        axes[i].set_title(model_name)
        axes[i].axis("off")

        del img, attr, ig
        xm = xm.to("cpu")
        torch.cuda.empty_cache()

    fig.savefig(os.path.join(out_dir, "ig_across_models.png"), bbox_inches="tight")
    plt.show()


In [17]:
# XAI models (same as explainability notebook)
models_xai = {
    "resnet50": get_resnet50_model("cpu"),   # load on CPU first
    "mobilenet_v3": get_mobilenet_v3_model("cpu"),
    "vit": get_vit_model("cpu"),
    "pvt": get_pvt_model("cpu"),
    "clip_vision": get_clip_vision_model("cpu")
}



Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/177 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_at

In [18]:
def compute_embedding(model, transform, path):
    img = transform(Image.open(path)).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(img)
    return logits.cpu().numpy().squeeze()

def retrieve_top_k(query_emb, all_embs, k=50):
    sims = cosine_similarity(query_emb.reshape(1,-1), all_embs)[0]
    idxs = np.argsort(-sims)[:k]
    return idxs, sims[idxs]


In [19]:
def show_visualization(model_name, query_path, heatmap, overlay, retrieved_paths):
    # Create a 2×3 grid: top row (query, heatmap, overlay), bottom row (top‑5 retrievals)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(model_name, fontsize=14)

    # --- Top row ---
    axes[0, 0].imshow(Image.open(query_path))
    axes[0, 0].set_title("Query")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(heatmap, cmap="hot")
    axes[0, 1].set_title("IG Heatmap")
    axes[0, 1].axis("off")

    axes[0, 2].imshow(overlay)
    axes[0, 2].set_title("Overlay")
    axes[0, 2].axis("off")

    # --- Bottom row: top‑5 retrieved images ---
    for i in range(3):
        if i < len(retrieved_paths):
            axes[1, i].imshow(Image.open(retrieved_paths[i]))
            axes[1, i].set_title(f"Top {i+1}")
            axes[1, i].axis("off")

    # Add extra retrieved images below if you want more than 3
    if len(retrieved_paths) > 3:
        fig2, axes2 = plt.subplots(1, 2, figsize=(12, 5))
        for i in range(3, 5):
            axes2[i-3].imshow(Image.open(retrieved_paths[i]))
            axes2[i-3].set_title(f"Top {i+1}")
            axes2[i-3].axis("off")
        plt.show()

    plt.tight_layout()
    plt.show()


In [20]:
def overlay_heatmap(image, heatmap, alpha=0.5):
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    heatmap = plt.cm.jet(heatmap)[..., :3]
    image = np.array(image.resize((heatmap.shape[1], heatmap.shape[0]))) / 255
    return (alpha * heatmap + (1 - alpha) * image)


# Execute

In [ ]:
for model_name in VISION_MODELS:

    print(f"\n=== Stress Test for {model_name} ===")

    # Load embeddings
    X_norm = load_embeddings(model_name)

    # Load indexation model (CPU)
    idx_model, idx_transform = load_indexation_model(model_name)
    idx_model = idx_model.to("cpu")

    # Load XAI model (GPU)
    xai_model, xai_transform = get_resnet50_model(device) if model_name == "resnet50" else \
                            get_mobilenet_v3_model(device) if model_name == "mobilenet_v3" else \
                            get_vit_model(device) if model_name == "vit" else \
                            get_pvt_model(device) if model_name == "pvt" else \
                            get_clip_vision_model(device)

    ig = IntegratedGradients(xai_model)


    torch.cuda.empty_cache()

    for sp in stress_paths:

        # -----------------------------
        # 1. Compute embedding (CPU)
        # -----------------------------
        img_idx = idx_transform(Image.open(sp)).unsqueeze(0)
        with torch.no_grad():
            out = idx_model(img_idx)

            if hasattr(out, "last_hidden_state"):
                q_emb = out.last_hidden_state[:,0,:].numpy().squeeze()
            elif hasattr(out, "pooler_output"):
                q_emb = out.pooler_output.numpy().squeeze()
            else:
                q_emb = out.numpy().squeeze()

        # -----------------------------
        # 2. Retrieval
        # -----------------------------
        idxs, sims = retrieve_top_k(q_emb, X_norm, k=50)
        retrieved_paths = [IMAGE_PATHS[i] for i in idxs]

        # -----------------------------
        # 3. IG for query (GPU)
        # -----------------------------
        img_xai = xai_transform(Image.open(sp)).unsqueeze(0).to(device)
        ig_attr = ig.attribute(img_xai, target=0)

        heatmap = ig_attr.squeeze().detach().cpu().numpy().mean(axis=0)
        overlay = overlay_heatmap(Image.open(sp), heatmap)

        del img_xai, ig_attr
        torch.cuda.empty_cache()

        # -----------------------------
        # 4. Save query results
        # -----------------------------
        img_name = os.path.basename(sp).replace(".jpg", "")
        out_dir = os.path.join(BASE_DIR, "Stress_Test", "Flickr8k", model_name, img_name)
        os.makedirs(out_dir, exist_ok=True)

        Image.open(sp).save(os.path.join(out_dir, "query.jpg"))
        plt.imsave(os.path.join(out_dir, "heatmap.png"), heatmap, cmap="hot")
        plt.imsave(os.path.join(out_dir, "overlay.png"), overlay)

        # -----------------------------
        # 5. IG per retrieved image (Top‑5)
        # -----------------------------
        retrieved_heatmaps = []
        retrieved_overlays = []

        for rp in retrieved_paths[:5]:
            # Move XAI model to GPU for each IG call
            xai_model = xai_model.to(device)
            h, o = compute_ig_for_image(xai_model, xai_transform, rp)

            retrieved_heatmaps.append(h)
            retrieved_overlays.append(o)

            idx = len(retrieved_heatmaps)
            plt.imsave(os.path.join(out_dir, f"retrieved_{idx}_heatmap.png"), h, cmap="hot")
            plt.imsave(os.path.join(out_dir, f"retrieved_{idx}_overlay.png"), o)

            torch.cuda.empty_cache()

        # -----------------------------
        # 6. IG across models (call function)
        # -----------------------------
        compare_models_ig_across_models(sp, models_xai, out_dir)

        # -----------------------------
        # 7. Visualizations
        # -----------------------------
        show_query_heatmap_overlay(model_name, sp, heatmap, overlay)
        show_topk_retrievals(retrieved_paths[:5], sims[:5])
        show_retrieved_ig(retrieved_paths[:5], retrieved_heatmaps, retrieved_overlays, sims[:5])

        del heatmap, overlay, retrieved_heatmaps, retrieved_overlays
        torch.cuda.empty_cache()

    # Move XAI model back to CPU
    xai_model = xai_model.to("cpu")
    torch.cuda.empty_cache()



=== Stress Test for resnet50 ===
